In [15]:
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from dotenv import load_dotenv
from langchain_groq import ChatGroq

In [5]:
load_dotenv()

True

In [16]:
model = ChatGroq(model='qwen/qwen3.8-27b')

In [7]:
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", output_dimensionality=100)

In [11]:
documents = [
    "Drinking enough water helps maintain hydration and supports normal body functions.",
    "Regular exercise improves cardiovascular health and strengthens muscles.",
    "Getting seven to nine hours of sleep supports physical recovery and mental well-being.",
    "A balanced diet includes fruits, vegetables, proteins, whole grains, and healthy fats.",
    "Meditation and deep breathing can help reduce stress and improve mental relaxation.",
    "Walking for thirty minutes daily can improve fitness and support heart health.",
    "Excessive sugar consumption may increase the risk of obesity and type 2 diabetes.",
    "Vitamin D plays an important role in maintaining healthy bones and muscles.",
    "Maintaining good posture can reduce unnecessary strain on the neck and back.",
    "Regular health checkups help identify potential health problems at an early stage.",
    "Consistent sleep schedules help regulate the body's internal biological clock.",
    "Yoga combines physical postures, breathing techniques, and mindfulness to support wellness.",
    "Solar panels convert sunlight into electrical energy for residential and commercial use.",
    "A balanced suspension system improves vehicle stability on uneven roads.",
    "Wind turbines generate renewable energy by converting wind power into electricity.",
    "Power distribution networks deliver electricity from generating stations to consumers.",
    "A well-balanced washing machine produces less vibration during high-speed spinning."
]

In [12]:
docs = []

for i, sentence in enumerate(documents):
    doc = Document(
        page_content=sentence,
        metadata={"id":i+1}
    )
    docs.append(doc)

In [13]:
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [14]:
similarity_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}
)

In [17]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k":5}),
    llm = model
)

In [18]:
query = "How to improve energy levels and maintain balance?"

In [19]:
similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

In [20]:
for i, doc in enumerate(similarity_results):
    print(f"--Result {i+1}--")
    print(doc.page_content)

print("*"*100)

for i, doc in enumerate(multiquery_results):
    print(f"--Result {i+1}--")
    print(doc.page_content)

--Result 1--
A well-balanced washing machine produces less vibration during high-speed spinning.
--Result 2--
A balanced suspension system improves vehicle stability on uneven roads.
--Result 3--
Drinking enough water helps maintain hydration and supports normal body functions.
--Result 4--
A balanced diet includes fruits, vegetables, proteins, whole grains, and healthy fats.
--Result 5--
Excessive sugar consumption may increase the risk of obesity and type 2 diabetes.
****************************************************************************************************
--Result 1--
Drinking enough water helps maintain hydration and supports normal body functions.
--Result 2--
Walking for thirty minutes daily can improve fitness and support heart health.
--Result 3--
Regular health checkups help identify potential health problems at an early stage.
--Result 4--
Regular exercise improves cardiovascular health and strengthens muscles.
--Result 5--
Consistent sleep schedules help regulate t